In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Week5Assignment") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Started Successfully")

Spark Started Successfully


Dataset Creation:

In [15]:
data = [
    (1, "John", "West", "Electronics", 500, 25, "Premium", "[email protected]", "2025-01-01", 101),
    (2, "Alice", "West", "Clothing", 300, 22, "Premium", "[email protected]", "2025-01-02", 101),
    (3, "Bob", "East", "Electronics", None, 35, "Basic", None, "2025-01-03", 102),
    (4, "", "West", "Electronics", 700, 28, "Premium", "[email protected]", "2025-01-01", 101),
    (1, "John", "West", "Electronics", 500, 25, "Premium", "[email protected]", "2025-01-01", 101)
]

columns = [
    "user_id", "username", "region", "product_category",
    "sale_amount", "age", "subscription",
    "email", "transaction_date", "store_id"
]

df = spark.createDataFrame(data, columns)

df.show()

+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+
|user_id|username|region|product_category|sale_amount|age|subscription|            email|transaction_date|store_id|
+-------+--------+------+----------------+-----------+---+------------+-----------------+----------------+--------+
|      1|    John|  West|     Electronics|        500| 25|     Premium|[email protected]|      2025-01-01|     101|
|      2|   Alice|  West|        Clothing|        300| 22|     Premium|[email protected]|      2025-01-02|     101|
|      3|     Bob|  East|     Electronics|       NULL| 35|       Basic|             NULL|      2025-01-03|     102|
|      4|        |  West|     Electronics|        700| 28|     Premium|[email protected]|      2025-01-01|     101|
|      1|    John|  West|     Electronics|        500| 25|     Premium|[email protected]|      2025-01-01|     101|
+-------+--------+------+----------------+-----------+---+------------+-

Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?
Ans. Traditional MapReduce has several limitations:

1) It relies heavily on disk I/O, which increases processing time.

2) Each stage writes intermediate results to disk before the next stage begins.

3) It is not suitable for iterative machine learning algorithms.

----> To overcome this spark is preferred because it solves these problems:

1) It uses in-memory processing.

2) It gives faster execution compared to MapReduce.

3) It supports machine learning.

Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.
Ans. Spark improves performance by storing intermediate computation results in RAM rather than writing them to disk after every operation. In iterative machine learning algorithms, the same dataset is processed multiple times. In MapReduce, each iteration requires reading data from disk again, which is slow. Spark keeps the data in memory and reuses it across iterations, resulting in much faster execution.

Q3. Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Week5").getOrCreate()

data = [
    (1, "2025-01-01", 500),
    (1, "2025-01-01", 500),
    (2, "2025-01-02", 700)
]

columns = ["user_id", "transaction_date", "amount"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

df_clean = df.dropDuplicates(["user_id", "transaction_date"])

print("After Removing Duplicates:")
df_clean.show()

Original Data:
+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|      1|      2025-01-01|   500|
|      1|      2025-01-01|   500|
|      2|      2025-01-02|   700|
+-------+----------------+------+

After Removing Duplicates:
+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|      1|      2025-01-01|   500|
|      2|      2025-01-02|   700|
+-------+----------------+------+



Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is
'West' and then group by product_category to find the average sale_amount.

In [19]:
df.filter(col("region") == "West") \
  .groupBy("product_category") \
  .avg("sale_amount") \
  .show()

ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

Q5: What is the difference between .na.drop() and .na.fill()? Provide a code
example of filling null values in a status column with the string 'Unknown'.

Ans..na.drop() removes rows containing null values.
.na.fill() replaces null values with a specified value.
Example: Filling missing values in a status column with 'Unknown'.

In [ ]:
df.na.fill("Unknown").show()

Q6: Write a query to find the total count of records for each city in a DataFrame, but only
for cities where the count is greater than 100.

In [ ]:
city_data = [
    ("Jaipur",),
    ("Jaipur",),
    ("Delhi",),
    ("Delhi",),
    ("Delhi",)
]

city_df = spark.createDataFrame(city_data, ["city"])

city_df.groupBy("city") \
       .count() \
       .filter(col("count") > 2) \
       .show()

Q7: How does the immutability of Spark DataFrames affect how you perform "data
cleaning" steps like dropping columns or renaming them?

Ans. Spark DataFrames are immutable, meaning they cannot be modified directly.
Operations such as dropping columns or renaming columns create a new DataFrame instead of changing the original one.
This helps maintain consistency and fault tolerance in distributed processing.

In [ ]:
df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
).show()

Q9: When cleaning a dataset, why is it often better to handle null values before
performing mathematical aggregations like sum() or avg()?
Ans.

Q10: Write the code to revise a column named raw_timestamp by casting it to a
TimestampType and renaming it to event_time.

In [ ]:
df = df.withColumn(
    "raw_timestamp",
    col("transaction_date").cast(TimestampType())
)

df = df.withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

df.show()

Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it
considered a wide transformation?

Ans. Shuffle is the process of redistributing data across different partitions during operations such as groupBy(), join(), and distinct().During a shuffle, Spark moves data between executors so that related records are brought together for processing.
It is called a wide transformation because data from multiple partitions is exchanged across the cluster, unlike narrow transformations where data remains within the same partition.

Q12: Write a code snippet that identifies and removes rows where the email column
contains null values OR the username is an empty string.

In [ ]:
df_clean = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

df_clean.show()

Q13: How do you use the .agg() function to calculate multiple statistics at once, such as
the min, max, and mean of the price column?

In [ ]:
df.agg(
    min("sale_amount").alias("Minimum Sale"),
    max("sale_amount").alias("Maximum Sale"),
    avg("sale_amount").alias("Average Sale")
).show()

Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true
when your source data contains messy or inconsistent date formats?

Ans. When inferSchema=True is used, Spark automatically determines the data type of each column.
If date values are stored in different formats such as:
01/01/2025
2025-01-01
Jan-01-2025
Spark may incorrectly infer the column type or treat some values as strings.This can lead to parsing errors, incorrect analysis, and data quality issues. Therefore, it is often better to define the schema explicitly when working with inconsistent data.

Q15: Write a final processing pipeline that:
1. Filters out duplicates.
2. Fills null prices with 0.
3. Groups by store_id to calculate total revenue.

In [ ]:
result = (
    df
    .dropDuplicates()
    .na.fill({"sale_amount": 0})
    .groupBy("store_id")
    .sum("sale_amount")
)

result.show()